# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and performing basic analysis on the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The data are described by a [Croissant schema](https://mlcommons.org/croissant/), available here:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and content using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Examine available record sets and fields. All entities are referenced by their `@id`s.

Let's enumerate the record sets, their fields, and columns as described by the Croissant schema.

In [ ]:
# List all record sets and their fields using their @id
record_sets = list(dataset.record_sets())
if not record_sets:
    print("No record sets found in the Croissant schema.")
else:
    for rs in record_sets:
        print(f"RecordSet: {rs['@id']}")
        if 'field' in rs:
            fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
            for f in fields:
                print(f"  Field: {f['@id']} - {f.get('name', f.get('@id'))}")
        else:
            print("  (No fields defined)")

In this dataset, depending on the version of the file, record sets may be empty in the schema root. However, the schema may define at least one RecordSet for the main data table.

**Let's load the record set directly by `@id`.**

To find all possible record sets or, if only one exists, use its `@id`. If the previous cell shows record sets, use that ID for `record_set_id` in the following sections.

In [ ]:
# Try to list all data records in the first available record set
record_sets = list(dataset.record_sets())
if not record_sets:
    print("No record sets found; please inspect the schema or dataset for tabular data.")
else:
    record_set_id = record_sets[0]['@id']
    count = 0
    print(f"First 3 records from RecordSet {record_set_id} (@id):")
    for rec in dataset.records(record_set=record_set_id):
        print(json.dumps(rec, indent=2))
        count += 1
        if count >= 3:
            break

## 3. Data Extraction
Load the table from the record set (referenced by `@id`) as a Pandas DataFrame.

In [ ]:
if not record_sets:
    print("No record sets to extract.")
else:
    # List of record set IDs (even if only one)
    record_set_ids = [rs['@id'] for rs in record_sets]
    dataframes = {}
    for rsid in record_set_ids:
        records = list(dataset.records(record_set=rsid))
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"Loaded {len(df)} records from RecordSet '@id': {rsid}")

    main_record_set_id = record_set_ids[0]
    print(f"Columns in main record set '@id' {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's perform simple filtering, normalization, and grouping for numeric columns. For illustration, we'll automatically pick the first numeric column.

In [ ]:
import numpy as np

if not record_sets:
    print("No data loaded for EDA.")
else:
    df = dataframes[main_record_set_id]
    # Try to identify numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) == 0:
        # Try to coerce to numbers for commonly named numeric fields
        possible_numeric = [col for col in df.columns if 'count' in col.lower() or 'mw' in col.lower() or 'number' in col.lower() or 'quantity' in col.lower() or 'abundance' in col.lower()]
        for col in possible_numeric:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) == 0:
        print("No numeric columns found for EDA.")
    else:
        numeric_field = numeric_cols[0]
        threshold = df[numeric_field].mean()  # Use mean as illustrative threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        print(filtered_df[[numeric_field, norm_col]].head())
        # Find a candidate group field (e.g. one with string/object dtype)
        candidate_groups = df.select_dtypes(include=[object]).columns
        group_field = candidate_groups[0] if len(candidate_groups) > 0 else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped data by '{group_field}':")
            print(grouped_df.head())
        else:
            print("No suitable group field for aggregation found.")

## 5. Visualization
Let's plot the distribution of the main numeric field and, if applicable, group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not record_sets or len(numeric_cols) == 0:
    print("No data available for plotting.")
else:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    if group_field:
        plt.figure(figsize=(10, 5))
        order = grouped_df.sort_values(numeric_field, ascending=False)[group_field].head(10)
        sns.barplot(data=grouped_df[grouped_df[group_field].isin(order)],
                    x=group_field, y=numeric_field, order=order)
        plt.title(f"Mean {numeric_field} by {group_field} (top 10)")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
We have loaded the FAIR² mass spectrometry dataset using its Croissant schema, explored available record sets and fields, extracted the main data table, and performed basic EDA and visualization. For detailed analysis, consult the Croissant schema documentation and dataset metadata to understand the domain meaning of each field.
